# Candle Streaks 03 — Does aggressive buy/sell flow predict how streaks start or stop?

**Historical exploratory / no AM-T-HAR-Blend modification.** Original five-minute *spot* price sign streaks and downloaded Binance USD-M futures **executed** aggressive-buy/total quote volumes are aligned on the **end of the just-completed five-minute candle**. Prediction target is the immediately following five-minute spot return: DOWN / ZERO / UP. No L2 order-book information is claimed. See [CS03 protocol](../../discussions/Candle_Streaks/003_chatgpt_trade_flow_streak_transitions_preregistration.md).

**Causality:** Binance minute `open_time` of 00:00–00:04 aggregates to the **00:05 close**. It cannot be used to forecast the already-ended 00:00–00:05 candle. Every fitted model here forecasts 00:05–00:10, or later. Zero returns break streaks; current flat bars can start new streaks. No arbitrary maximum streak age.

## 0 · Environment and test suite

In [ ]:
import os,sys,json,subprocess,time
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir(): ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Run notebook in repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import candle_streak_trade_flow as tf
cfg=tf.Specification()
SYNTHETIC=os.getenv('CS03_SYNTHETIC','0')=='1'
if not SYNTHETIC:
    from data_engineering.modeling import markov_lift_distributions as vd14
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='cs03-smoke-')
    OUT=Path(tmp.name)/'03'
else:
    OUT=ROOT/'notebooks'/'Candle_Streaks'/'03'
OUT.mkdir(parents=True,exist_ok=True)
print('Version:',tf.VERSION,'synthetic:',SYNTHETIC,'output:',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
check=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_candle_streak_trade_flow.py')],
                     cwd=str(ROOT),env=env,capture_output=True,text=True)
print(check.stdout[-2400:]);print(check.stderr[-1800:]);assert check.returncode==0, 'CS03 regression tests failed'


## 1 · Read historical spot streaks and existing Binance USD-M taker-buy data

The existing `tools/download_binance_um_klines.py` writes `<ASSET>USDT_um_1m.parquet` into `BINANCE_UM_DIR` or `<parent of DERIBIT_DATA_ROOT>/Binance`. `tools/download_price_history.py` provides spot five-minute closes. No new historical data are downloaded. Synthetic mode creates matched example input **without writing price data**.

In [ ]:
ASSET_DATA={};manifest=[];exclusions={}
if SYNTHETIC:
    rng=np.random.default_rng(2604)
    days=166
    n=days*288
    minutes=pd.date_range('2026-01-01',periods=n*5,freq='1min',tz='UTC')
    buy_fraction=np.clip(.5+.13*np.sin(np.arange(len(minutes))/13)+rng.normal(0,.08,len(minutes)),.04,.96)
    vol=rng.uniform(100,400,len(minutes))
    fake_minutes=pd.DataFrame({'open_time':minutes,'quote_volume':vol,
                 'taker_buy_quote_volume':vol*buy_fraction,'trades':rng.integers(1,8,len(minutes))})
    futures_flow=tf.one_minute_futures_flow(fake_minutes)
    flow_signal=futures_flow.flow_5m.fillna(0).to_numpy(float)
    # Synthetic future dependence is intentional for implementation verification ONLY.
    pseudo=(rng.uniform(size=n) < (.48+.15*np.tanh(flow_signal*3)))
    ret=np.where(pseudo,1.,-1.)*rng.uniform(.0002,.003,n)
    ret[rng.uniform(size=n)<.03]=0.
    state=futures_flow.state_time
    spot=pd.DataFrame({'state_time':state,'return_5m':ret})
    ASSET_DATA={'BTC':(spot,futures_flow)}
    raw_coverage='SYNTHETIC data (NOT real Binance history)'
else:
    rootstr=os.getenv('DERIBIT_DATA_ROOT')
    if not rootstr:
        raise RuntimeError('DERIBIT_DATA_ROOT not configured in .env or environment')
    root=Path(rootstr)
    hist=root/'history'/'price_5m'
    futroot=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else root.parent/'Binance'
    print('Spot:',hist,' Futures:',futroot)
    for a in cfg.assets:
        fp=futroot/f'{a}USDT_um_1m.parquet'
        if not fp.is_file():
            exclusions[a]=f'Missing downloaded futures parquet: {fp}'
            continue
        try:
            price,excluded=vd14.load_returns(hist,(a,),end=pd.Timestamp('2026-10-14T00:00:00Z'),min_days=365)
            sp=price.loc[price.underlying==a,['state_time','return_5m']].copy()
            s0=sp.state_time.min()-pd.Timedelta(days=2)
            s1=sp.state_time.max()+pd.Timedelta(minutes=5)
            ff=tf.read_futures_parquet(fp,start=s0,end=s1)
            ASSET_DATA[a]=(sp,ff)
        except Exception as exc:
            exclusions[a]=f'{type(exc).__name__}: {exc}'
    raw_coverage='Existing local spot 5m and Binance USD-M futures 1m files'
assert ASSET_DATA, f'No matched local input assets: {exclusions}'
print('Available assets:',sorted(ASSET_DATA),'excluded:',exclusions)


## 2 · Strict complete-day overlap and continuous signed streaks

We retain flat-origin bars (`sign=0, length=0`) to measure new streak initiation after a flat candle. The forecasting target is strictly the next five-minute close. Each evaluated day requires **288 valid spot returns and 288 matched complete 5×1m futures flow bars**.

In [ ]:
prepared={};splits={};coverage=[];quality=[]
for asset,(sp,ff) in ASSET_DATA.items():
    panel=tf.build_origin_panel(sp,ff,fast_bars=cfg.fast_bars,slow_bars=cfg.slow_bars,
                                flow_bars=cfg.flow_bars)
    dates=tf.full_days(panel)
    quality.append({'asset':asset,'source_spot_rows':len(sp),'futures_5m_buckets':len(ff),
                    'futures_valid_5m_buckets':int(ff.flow_valid.sum()),
                    'spot_zero_current':int((panel.sign==0).sum()),
                    'complete_overlap_days':len(dates),
                    'earliest_complete_date':str(dates.min()) if len(dates) else None,
                    'latest_complete_date':str(dates.max()) if len(dates) else None})
    if len(dates)<60:
        exclusions[asset]=f'Insufficient complete overlapping days: {len(dates)}'
        continue
    x,meta=tf.split_panel(panel,dates,cfg.train_fraction,cfg.valid_fraction)
    x['asset']=asset
    prepared[asset]=x;splits[asset]=meta
    for period,g in x.groupby('period'):
        coverage.append({'asset':asset,'period':period,'full_dates':g.date.nunique(),
                         'n_eligible_current':len(g),'n_scorable_next':int(g.scorable.sum()),
                         'n_current_flat':int((g.sign==0).sum()),
                         'n_next_flat_scored':int((g.scorable & g.next_sign.eq(0)).sum()),
                         'n_reversal_scored':int((g.scorable & g.outcome.eq('reverse_new_streak')).sum())})
print('Eligible:',list(prepared),'exclusions:',exclusions)
display(pd.DataFrame(quality));display(pd.DataFrame(coverage))
assert prepared, 'No asset has at least 60 complete overlapping days'
if not SYNTHETIC and not {'BTC','ETH'}.issubset(prepared):
    print('WARNING: BTC and ETH both required for the preregistered primary decision; report coverage only for missing assets.')


## 3 · Fit TRAIN-only models and evaluate VALID / TEST

Pre-registered models: IID, sign-only, sign+exact-log-age, sign+age+volatility, sign+age+trade-flow and sign+age+volatility+trade-flow. No hyperparameter tuning on holdout. Primary incremental comparison: `sign_streak_vol_flow` minus `sign_streak_vol` on **unconditional DOWN/ZERO/UP** log loss. Secondary: binary UP/DOWN log loss conditional on a nonzero immediate next return.

In [ ]:
all_scores=[];all_boot=[];all_transitions=[];all_coeff=[];all_prob_groups=[];all_null=[];models={};flow_cuts={}
for asset,x in prepared.items():
    tr=x.loc[x.period=='train'];fit=tf.fit_models(tr,C=cfg.logistic_C)
    cuts=tf.fit_flow_cuts(tr)
    models[asset]=fit;flow_cuts[asset]=cuts
    all_transitions.append(tf.observed_transitions(x,cuts).assign(asset=asset))
    for name,(terms,scaler,model) in fit['models'].items():
        for feature,coefrow in zip(terms,model.coef_.T):
            all_coeff.append({'asset':asset,'model':name,'feature':feature,
                              **{f'class_{int(c)}':float(v) for c,v in zip(model.classes_,coefrow)}})
    for period in ('valid','test'):
        ev=x.loc[(x.period==period)&x.scorable].copy()
        pred=tf.predict_models(ev,fit)
        summary,day=tf.score_models(ev,pred)
        all_scores.append(summary.assign(asset=asset,period=period))
        all_boot.append(tf.paired_day_bootstrap(day,B=cfg.bootstrap_reps if not SYNTHETIC else 80,
                                               seed=cfg.seed).assign(asset=asset,period=period))
        # Frozen predicted probabilities averaged in each observed flow/streak cell.
        ev['flow_tier']=np.searchsorted(cuts,ev.flow_5m.to_numpy(float),side='right')
        ev['age_group']=pd.cut(ev.length,bins=[-1,0,1,2,3,5,8,np.inf],
                       labels=['0 flat','1','2','3','4-5','6-8','9+'])
        p=pred['sign_streak_vol_flow']
        ev['model_next_up']=p[:,2];ev['model_next_down']=p[:,0];ev['model_next_zero']=p[:,1]
        groups=ev.groupby(['sign','age_group','flow_tier'],observed=True).agg(
            n=('next_sign','size'),p_fit_up=('model_next_up','mean'),
            p_fit_down=('model_next_down','mean'),p_fit_zero=('model_next_zero','mean'),
            p_observed_up=('next_sign',lambda z:float((z==1).mean())),
            p_observed_down=('next_sign',lambda z:float((z==-1).mean())),
            p_observed_zero=('next_sign',lambda z:float((z==0).mean()))).reset_index()
        all_prob_groups.append(groups.assign(asset=asset,period=period))
    te=x.loc[(x.period=='test')&x.scorable]
    all_null.append({'asset':asset,**tf.shuffle_test_flow(te,fit,B=cfg.shuffle_reps if not SYNTHETIC else 5,seed=cfg.seed)})
scores=pd.concat(all_scores,ignore_index=True)
bootstrap=pd.concat(all_boot,ignore_index=True)
transitions=pd.concat(all_transitions,ignore_index=True)
probability_cells=pd.concat(all_prob_groups,ignore_index=True)
coefficients=pd.DataFrame(all_coeff);null=pd.DataFrame(all_null)
display(scores[(scores.period=='test')][['asset','model','n','threeclass_logloss','binary_logloss','observed_zero']])


## 4 · How flow relates to new streaks and streak termination

View raw conditional **transition frequencies** across TRAIN, VALID, and TEST. Flow tertile thresholds are frozen from TRAIN, while age groups are for readability only. A `−4` streak is a running four-down sequence; its `reverse_new_streak` outcome means next bar is UP. A `0 flat` origin's `start_up/down_after_flat` means a new nonzero streak begins on the next bar.

In [ ]:
display(transitions[(transitions.period=='test')&(transitions.n>=50)][
    ['asset','sign','age_group','flow_tier','n','p_continue','p_reverse','p_stop_flat',
     'p_start_up_after_flat','p_start_down_after_flat']].head(42))
print('Flow tertile thresholds fitted on TRAIN:',flow_cuts)
display(probability_cells[(probability_cells.period=='test')&(probability_cells.n>=50)].head(25))


## 5 · Paired-day bootstrap, null, decision

A favorable **unconditional three-class** result does not establish directional edge on nonzero candles. Binary up/down scores are separately reported. The null destroys alignment of executed trade flow to outcomes without retraining; it is a diagnostic, not formal generative significance.

In [ ]:
test_boot=bootstrap[(bootstrap.period=='test')]
display(test_boot[(test_boot.metric.isin(['threeclass_logloss','binary_logloss']))&
                    (test_boot.model=='sign_streak_vol_flow')][
 ['asset','metric','reference','mean_day_delta','ci025','ci975','days']])
display(null)
primary=[]
for asset in ('BTC','ETH'):
    g=test_boot[(test_boot.asset==asset)&(test_boot.metric=='threeclass_logloss')&
                (test_boot.model=='sign_streak_vol_flow')]
    ref=g[g.reference=='sign_streak_vol']
    primary.append({'asset':asset,'available':len(ref)==1,
        'delta':float(ref.iloc[0].mean_day_delta) if len(ref) else None,
        'upper95':float(ref.iloc[0].ci975) if len(ref) else None,
        'passed':bool(len(ref)==1 and ref.iloc[0].ci975<0)})
verdict='POTENTIAL_THREECLASS_FLOW_GAIN' if all(x['passed'] for x in primary) else 'NO_CONFIRMED_THREECLASS_FLOW_GAIN'
if not {'BTC','ETH'}.issubset(prepared):verdict='INSUFFICIENT_PRIMARY_ASSET_COVERAGE'
if SYNTHETIC:verdict='SYNTHETIC_SMOKE_NO_MARKET_INFERENCE'
decision={'status':'HISTORICAL_EXPLORATORY' if not SYNTHETIC else 'SYNTHETIC',
   'verdict':verdict,'primary':primary,'directional_gain_claimed':False,
   'AM_T_HAR_Blend_modified':False,'note':'Separate conditional binary scoring required before direction claims. Futures USD-M flow vs spot returns.'}
print(json.dumps(decision,indent=2))


## 6 · Save compact aggregate outputs only

In [ ]:
artifacts={
 '03_quality.csv':pd.DataFrame(quality),
 '03_coverage.csv':pd.DataFrame(coverage),
 '03_scores.csv':scores,
 '03_paired_day_bootstrap.csv':bootstrap,
 '03_flow_streak_transitions.csv':transitions,
 '03_frozen_probability_cells.csv':probability_cells,
 '03_model_coefficients.csv':coefficients,
 '03_flow_shuffle_null.csv':null,
}
for name,df in artifacts.items():
    df.to_csv(OUT/name,index=False)
json_out={
 '03_manifest.json':{'module':'CS03','registration':tf.registration(),
    'source':raw_coverage,'spot_source':'Binance spot 5m price closes',
    'flow_source':'Binance USD-M perpetual 1m taker-buy quote volume (NOT L2 OBI)',
    'assets_evaluated':list(prepared),'excluded':exclusions,
    'split':splits,'flow_tertiles_train':flow_cuts,
    'synthetic':SYNTHETIC},
 '03_decision.json':decision,
}
for name,data in json_out.items():
    (OUT/name).write_text(json.dumps(data,indent=2,default=str),encoding='utf-8')
print('Saved',len(artifacts)+len(json_out),'aggregates in',OUT)
assert all((OUT/name).is_file() for name in list(artifacts)+list(json_out))


## Interpretation checklist

- **Executed aggressive buying/selling ≠ resting book OBI.** USD-M futures order flow is joined to Binance *spot* close-to-close streaks; these are different instruments and markets.
- **Same-bar correlation does not establish a forecast.** Predictors include at most data through current 5m close; targets start immediately afterward.
- TEST outcome tables are retrospective, not tradable TRAIN-fitted matrix estimates. The saved `03_frozen_probability_cells.csv` summarizes TEST predictions **from the TRAIN-fitted model**, with corresponding test labels for evaluation; choosing cells by their TEST performance invalidates inference.
- A zero spot return might occur with substantial USD-M futures trading; zero-origin rows are included rather than erased, and next-flat outcomes are scored in the unconditional three-class model.
- Historical returns overlap prior CS research, so even the TEST segment is exploratory. No signal is integrated into AM-T-HAR-Blend without prospective and execution-aware validation.